## Exercise 1: Access Control

You are a data engineer at **NorthMart Retail**, a national supermarket chain. Your team stores customer and transaction data in Unity Catalog. Before any analyst can run queries, you must first create the catalog, the schema, and populate a customer table — then grant the `retail-analysts` group the permissions they need.

# > ⚠️ Before starting this exercise, ensure you have created the `retail-analysts` group in Databricks and added your own user account to it. See the lab setup instructions.

%md
## **Before starting the lab, you need a Unity Catalog volume to store the file**. Follow these steps to create one:

- In the Databricks workspace sidebar, click Catalog.
- In the Catalog Explorer,right side click on + icon and create a Catalog ( DP750_Lab_04 ), type =standard & default storage.
- Then Create expand Catalog and generate a new schema (security_lab) by clicking on + icon and create Schema.
- Click the ⋮ menu next to default, then select Create > Volume. (security_lab)
- Enter lab_data as the volume name, leave the type as Managed, and click Create.

### Task 1.1 — Create the  customers table

The cell below sets up all the objects needed for this exercise:
- Creates a managed Delta table `retail_catalog.security_lab.customers` and populates it with six sample customer records. These rows will be used throughout the rest of this exercise to demonstrate row filtering and column masking.

| customer_id | customer_name   | email                   | region | total_spend | tier     |
|-------------|-----------------|-------------------------|--------|-------------|----------|
| 1           | Alice Johnson   | alice@northmart.com     | North  | 5200.00     | Standard |
| 2           | Bob Martinez    | bob@northmart.com       | South  | 8100.00     | Premium  |
| 3           | Carol Lee       | carol@northmart.com     | North  | 3400.00     | Standard |
| 4           | David Chen      | david@northmart.com     | East   | 7800.00     | Premium  |
| 5           | Eve Patel       | eve@northmart.com       | West   | 6200.00     | Standard |
| 6           | Frank Kim       | frank@northmart.com     | South  | 4100.00     | Standard |

Run the cell to provision these objects.

In [0]:
%sql
CREATE OR REPLACE TABLE dp750_lab_04.security_lab.customers AS
SELECT * FROM VALUES
  (1, 'Alice Johnson', 'alice@northmart.com', 'North', 5200.00, 'Standard'),
  (2, 'Bob Martinez',  'bob@northmart.com',   'South', 8100.00, 'Premium'),
  (3, 'Carol Lee',     'carol@northmart.com', 'North', 3400.00, 'Standard'),
  (4, 'David Chen',    'david@northmart.com', 'East',  7800.00, 'Premium'),
  (5, 'Eve Patel',     'eve@northmart.com',   'West',  6200.00, 'Standard'),
  (6, 'Frank Kim',     'frank@northmart.com', 'South', 4100.00, 'Standard')
  AS t(customer_id, customer_name, email, region, total_spend, tier);

num_affected_rows,num_inserted_rows


### Task 1.2 — Grant permissions to the retail-analysts group

The `retail-analysts` group needs to query the `customers` table. Grant the necessary privileges at all required levels of the Unity Catalog hierarchy:

- `USE CATALOG` on `lab4`
- `USE SCHEMA` on `lab4.security_lab`
- `SELECT` on `lab4.security_lab.customers`



# Grant permissions to the retail-analysts group

spark.sql("GRANT USE CATALOG ON CATALOG lab4 TO `retail-analysts`")

Explanation:

- `GRANT` is used to give permissions to a user or group.
- `USE CATALOG` allows the group to access and browse the `lab4` catalog.
- `retail-analysts` is the group receiving the permission.

Purpose:
Allow retail analysts to access the `lab4` catalog.

--------------------------------------------------

spark.sql("GRANT USE SCHEMA ON SCHEMA lab4.security_lab TO `retail-analysts`")

Explanation:

- `USE SCHEMA` allows users to access objects inside the schema.
- `lab4.security_lab` is the schema being accessed.
- Without this permission, users cannot work with tables inside the schema.

Purpose:
Allow retail analysts to access the `security_lab` schema.

--------------------------------------------------

spark.sql("GRANT SELECT ON TABLE lab4.security_lab.customers TO `retail-analysts`")

Explanation:

- `SELECT` permission allows users to read data from a table.
- `lab4.security_lab.customers` is the table being shared.
- Users can run queries like `SELECT * FROM customers`.

Purpose:
Allow retail analysts to view customer data without modifying it.

--------------------------------------------------

Overall Purpose:

These commands implement role-based access control (RBAC) by giving the `retail-analysts` group permission to:

1. Access the `lab4` catalog.
2. Access the `security_lab` schema.
3. Read data from the `customers` table.

This follows the hierarchy:
Catalog Permission → Schema Permission → Table Permission

In [0]:
# Grant permissions to the retail-analysts group
spark.sql("GRANT USE CATALOG ON CATALOG dp750_lab_04 TO `new_group`")
spark.sql("GRANT USE SCHEMA ON SCHEMA dp750_lab_04.security_lab TO `new_group`")
spark.sql("GRANT SELECT ON TABLE dp750_lab_04.security_lab.customers TO `new_group`")

DataFrame[]

### Task 1.3 — Verify the granted permissions

Verify that the privileges were correctly assigned. Use `SHOW GRANTS` to display all privileges on the `customers` table.



# Show all grants on the customers table to verify permissions

display(spark.sql("SHOW GRANTS ON TABLE lab4.security_lab.customers"))

Explanation:

- `SHOW GRANTS` displays all permissions assigned to the specified table.
- `ON TABLE lab4.security_lab.customers` specifies the table whose permissions we want to check.
- Spark returns a result containing users/groups and their granted privileges.
- `spark.sql()` executes the SQL command.
- `display()` renders the result in a Databricks table format for easy viewing.

Purpose:
Verify that the correct permissions (such as SELECT access for `retail-analysts`) have been successfully granted to the `customers` table.

--------------------------------------------------

Overall Purpose:

This command is used to audit and confirm table-level access permissions by showing who has access and what actions they are allowed to perform on the `customers` table.

In [0]:
# Show all grants on the customers table to verify permissions
display(spark.sql("SHOW GRANTS ON TABLE dp750_lab_04.security_lab.customers"))

Principal,ActionType,ObjectType,ObjectKey
new_group,SELECT,TABLE,dp750_lab_04.security_lab.customers


## Exercise 2: Row Filtering

NorthMart Retail's regional analysts should only see customer records for their own region. You will implement a **row filter function** that restricts rows based on who is querying the table.

Since you cannot create multiple user accounts, you will write a filter that shows only `North` region rows to everyone *except* for your own user account, which sees all rows. This simulates what a regional analyst would experience.

### Task 2.1 — Create a row filter function

Create a SQL function `retail_catalog.security_lab.region_filter` that takes a `region` parameter of type `STRING` and returns `TRUE` if:
- the `region` is `'North'`, **OR**
- the current user is your own account (replace `your@email.com` with your actual email).

This means: unless you are the privileged user, only `North` region rows will be visible.




# Create the region_filter function in retail_catalog.security_lab

spark.sql("""
    CREATE OR REPLACE FUNCTION lab4.security_lab.region_filter(region STRING)
    RETURNS BOOLEAN
    RETURN region = 'North' OR current_user() = 'admin@vbhvprksh1outlook.onmicrosoft.com'
""")

Explanation:

- `CREATE OR REPLACE FUNCTION` creates a new function or replaces it if it already exists.
- `lab4.security_lab.region_filter` is the function name.
- `(region STRING)` defines an input parameter named `region` of type STRING.
- `RETURNS BOOLEAN` means the function will return either `TRUE` or `FALSE`.

--------------------------------------------------

RETURN region = 'North'

Explanation:

- Checks whether the provided region is `North`.
- Returns `TRUE` if the region is North.
- Returns `FALSE` for other regions.

Purpose:
Allow access only to records belonging to the North region.

--------------------------------------------------

OR current_user() = 'admin@vbhvprksh1outlook.onmicrosoft.com'

Explanation:

- `current_user()` returns the email/identity of the user running the query.
- Checks whether the current user is the specified admin user.
- If the user is the admin, the function returns `TRUE` regardless of region.

Purpose:
Give the admin full access to all records.

--------------------------------------------------

Overall Purpose:

This function is used for row-level security. It returns:

- `TRUE` for records where `region = 'North'`
- `TRUE` for the admin user on all records
- `FALSE` for other regions and users

This allows data access to be controlled based on both the row's region and the user's identity.

In [0]:
# Create the region_filter function in retail_catalog.security_lab
# Replace 'your@email.com' with your actual Databricks user email
spark.sql("""
    CREATE OR REPLACE FUNCTION dp750_lab_04.security_lab.region_filter(region STRING)
    RETURNS BOOLEAN
    RETURN region = 'North' OR current_user() = 'admin@vbhvprksh1outlook.onmicrosoft.com'
""")

DataFrame[]

In [0]:
%sql
Select * from dp750_lab_04.security_lab.customers

customer_id,customer_name,email,region,total_spend,tier
1,Alice Johnson,alice@northmart.com,North,5200.00,Standard
2,Bob Martinez,bob@northmart.com,South,8100.00,Premium
3,Carol Lee,carol@northmart.com,North,3400.00,Standard
4,David Chen,david@northmart.com,East,7800.00,Premium
5,Eve Patel,eve@northmart.com,West,6200.00,Standard
6,Frank Kim,frank@northmart.com,South,4100.00,Standard


### Task 2.2 — Apply the row filter to the customers table

Attach the `region_filter` function to the `customers` table so that it is automatically applied to every query against that table.


# Apply the region_filter function as a row filter on the region column of the customers table

spark.sql("""
    ALTER TABLE lab4.security_lab.customers
    SET ROW FILTER lab4.security_lab.region_filter ON (region)
""")

Explanation:

- `ALTER TABLE` modifies an existing table.
- `lab4.security_lab.customers` is the table being modified.
- `SET ROW FILTER` applies a row-level security rule to the table.
- `lab4.security_lab.region_filter` is the function that will determine which rows are visible.
- `ON (region)` passes the `region` column value to the `region_filter` function for each row.

--------------------------------------------------

How it works:

For every row in the `customers` table:

- The value of the `region` column is sent to `region_filter(region)`.
- If the function returns `TRUE`, the row is shown.
- If the function returns `FALSE`, the row is hidden.

Example:

- Region = 'North' → Function returns TRUE → Row is visible.
- Region = 'South' → Function returns FALSE → Row is hidden.
- Admin user → Function returns TRUE for all rows → All rows are visible.

--------------------------------------------------

Purpose:

Apply row-level security to the `customers` table so that users only see rows they are authorized to access, while the admin retains access to all records.

--------------------------------------------------

Overall Purpose:

This command enforces data security at the row level by attaching the `region_filter` function to the `customers` table. The filter is automatically applied whenever the table is queried.

In [0]:
# Apply the region_filter function as a row filter on the region column of the customers table
spark.sql("""
    ALTER TABLE dp750_lab_04.security_lab.customers
    SET ROW FILTER dp750_lab_04.security_lab.region_filter ON (region)
""")

DataFrame[]

### Task 2.3 — Test the row filter

Query the `customers` table. Because your own email is listed as the privileged user in the filter function, you should see **all 6 rows**.

If you had logged in as someone else (not your email), you would only see the 2 `North` region rows.



%sql

SELECT current_user();

Explanation:

- `current_user()` returns the email/identity of the user currently running the query.
- This helps verify which user's permissions and row filters will be applied.
- Since the row filter uses `current_user()`, the query result depends on who is logged in.

Purpose:
Confirm the current user before testing row-level security.

--------------------------------------------------

-- TODO: Query lab4.security_lab.customers and observe the results

Example:

SELECT * 
FROM lab4.security_lab.customers;

Explanation:

- Queries all records from the `customers` table.
- The row filter attached to the table is automatically applied.
- Users will only see rows that satisfy the `region_filter` function.
- The admin user will see all rows.

Expected Result:

- Regular users → Only rows where `region = 'North'`.
- Admin user (`admin@vbhvprksh1outlook.onmicrosoft.com`) → All customer rows.

Purpose:
Test and verify that the row-level security policy is working correctly.

--------------------------------------------------

Overall Purpose:

1. Identify the current user.
2. Query the `customers` table.
3. Verify that the row filter restricts data visibility based on the user's identity.

In [0]:
%sql
-- Verify the current user
SELECT current_user();

-- TODO: Query retail_catalog.security_lab.customers and observe the results

current_user()
admin@vbhvprksh1outlook.onmicrosoft.com


➡️ Tip: update the email address in the row filter function you created earlier, and test again. Observe the output when querying the customers table.

### Task 2.4 — Remove the row filter

Now remove the row filter from the `customers` table. After removing it, query the table again to confirm **all 6 rows** are now returned regardless of user context.


# Remove the row filter from the customers table

spark.sql("""
    ALTER TABLE lab4.security_lab.customers
    DROP ROW FILTER
""")

Explanation:

- `ALTER TABLE` is used to modify an existing table.
- `lab4.security_lab.customers` is the table being modified.
- `DROP ROW FILTER` removes the row-level security policy currently applied to the table.

--------------------------------------------------

What happens after this?

- The `region_filter` function is no longer enforced.
- Queries against the `customers` table will return all rows.
- Data visibility will depend only on table permissions (such as SELECT), not on row-level filtering.

--------------------------------------------------

Purpose:

Remove the row-level security rule from the `customers` table.

--------------------------------------------------

Overall Purpose:

This command disables row-level security by removing the attached row filter, allowing authorized users to see all records in the table.

In [0]:
%sql
Select * from dp750_lab_04.security_lab.customers

customer_id,customer_name,email,region,total_spend,tier
1,Alice Johnson,alice@northmart.com,North,5200.00,Standard
2,Bob Martinez,bob@northmart.com,South,8100.00,Premium
3,Carol Lee,carol@northmart.com,North,3400.00,Standard
4,David Chen,david@northmart.com,East,7800.00,Premium
5,Eve Patel,eve@northmart.com,West,6200.00,Standard
6,Frank Kim,frank@northmart.com,South,4100.00,Standard


In [0]:
# Remove the row filter from the customers table
spark.sql("""
    ALTER TABLE dp750_lab_04.security_lab.customers
    DROP ROW FILTER
""")

DataFrame[]

%sql

ALTER TABLE lab4.security_lab.customers
DROP ROW FILTER;

Explanation:

- `ALTER TABLE` modifies an existing table.
- `DROP ROW FILTER` removes the row-level security policy attached to the table.
- After this command, the `region_filter` function is no longer applied when querying the table.

Purpose:
Disable row-level security on the `customers` table.

--------------------------------------------------

SELECT * FROM lab4.security_lab.customers;

Explanation:

- `SELECT *` retrieves all columns and all rows from the table.
- Since the row filter has been removed, no rows are hidden based on region or user identity.
- Any user with `SELECT` permission can view all records.

Purpose:
Verify that the row filter was successfully removed.

--------------------------------------------------

Expected Result:

- All 6 customer records should be returned.
- No filtering based on the `region` column or `current_user()` will occur.

--------------------------------------------------

Overall Purpose:

1. Remove the row-level security policy.
2. Query the table again.
3. Confirm that all 6 customer records are now visible.

In [0]:
%sql ALTER TABLE dp750_lab_04.security_lab.customers
DROP ROW FILTER;

-- Query the table to confirm all 6 rows are visible
SELECT * FROM dp750_lab_04.security_lab.customers;

customer_id,customer_name,email,region,total_spend,tier
1,Alice Johnson,alice@northmart.com,North,5200.00,Standard
2,Bob Martinez,bob@northmart.com,South,8100.00,Premium
3,Carol Lee,carol@northmart.com,North,3400.00,Standard
4,David Chen,david@northmart.com,East,7800.00,Premium
5,Eve Patel,eve@northmart.com,West,6200.00,Standard
6,Frank Kim,frank@northmart.com,South,4100.00,Standard


## Exercise 3: Column Masking

Customer email addresses are PII and must be masked for users who aren't privileged. You will create a **column mask function** that hides the full email address for most users, applying it to the `email` column of the `customers` table.

### Task 3.1 — Create a column mask function

Create a SQL function `retail_catalog.security_lab.mask_email` that takes an `email` parameter of type `STRING` and:
- Returns the **full email** if the current user is your own account.
- Returns a **masked value** (e.g. `'***@***.com'`) for all other users.



# Create the mask_email function in retail_catalog.security_lab

spark.sql("""
    CREATE OR REPLACE FUNCTION lab4.security_lab.mask_email(email STRING)
    RETURNS STRING
    RETURN CASE WHEN current_user() = 'admin@vbhvprksh1outlook.onmicrosoft.com' THEN email ELSE '***@***.com' END
""")

Explanation:

- `CREATE OR REPLACE FUNCTION` creates a new function or updates it if it already exists.
- `lab4.security_lab.mask_email` is the function name.
- `(email STRING)` defines an input parameter called `email`.
- `RETURNS STRING` means the function returns a text value.

--------------------------------------------------

CASE WHEN current_user() = 'admin@vbhvprksh1outlook.onmicrosoft.com'

Explanation:

- `current_user()` returns the user executing the query.
- Checks whether the current user is the admin.

Purpose:
Identify users who are allowed to see the actual email addresses.

--------------------------------------------------

THEN email

Explanation:

- If the current user is the admin, the original email value is returned.

Example:

alice@northmart.com

Purpose:
Allow admins to view unmasked email addresses.

--------------------------------------------------

ELSE '***@***.com'

Explanation:

- For all other users, a masked value is returned instead of the real email.

Example:

***@***.com

Purpose:
Hide sensitive email information from non-admin users.

--------------------------------------------------

Overall Purpose:

This function implements column-level security by masking email addresses. The admin sees real email values, while all other users see a masked version to protect sensitive customer information.

In [0]:
# Create the mask_email function in retail_catalog.security_lab
# Replace 'your@email.com' with your actual Databricks user email
spark.sql("""
    CREATE OR REPLACE FUNCTION dp750_lab_04.security_lab.mask_email(email STRING)
    RETURNS STRING
    RETURN CASE WHEN current_user() = 'admin@vbhvprksh1outlook.onmicrosoft.com' THEN email ELSE '***@***.com' END
""")

DataFrame[]

%sql

CREATE OR REPLACE FUNCTION lab4.security_lab.mask_email(email STRING)
RETURNS STRING
RETURN CASE WHEN current_user() = 'admin@vbhvprksh1outlook.onmicrosoft.com' THEN email ELSE '***@***.com' END;

Explanation:

- `%sql` tells Databricks to execute the command as SQL.
- `CREATE OR REPLACE FUNCTION` creates a new function or replaces the existing one.
- `lab4.security_lab.mask_email` is the function name.
- `(email STRING)` defines an input parameter called `email`.
- `RETURNS STRING` indicates that the function returns a text value.

--------------------------------------------------

RETURN CASE WHEN ... THEN ... ELSE ... END

Explanation:

- `CASE WHEN` is a conditional statement similar to an IF-ELSE block.
- It checks the identity of the user running the query.

Condition:

current_user() = 'admin@vbhvprksh1outlook.onmicrosoft.com'

- `current_user()` returns the logged-in user's email.
- If the user is the admin, the condition is TRUE.

--------------------------------------------------

THEN email

Explanation:

- Returns the original email address.
- Admin users can see the actual email values.

Example:

alice@northmart.com

--------------------------------------------------

ELSE '***@***.com'

Explanation:

- Returns a masked value for all non-admin users.
- The real email address is hidden.

Example:

***@***.com

--------------------------------------------------

Purpose:

Create a masking function that protects customer email addresses by showing real values only to the admin while displaying masked values to other users.

--------------------------------------------------

Overall Purpose:

This function is used for column-level security (data masking). It ensures sensitive email data is protected while still allowing authorized users (admins) to view the original values.

In [0]:
%sql
CREATE OR REPLACE FUNCTION dp750_lab_04.security_lab.mask_email(email STRING)
RETURNS STRING
RETURN CASE WHEN current_user() = 'admin@vbhvprksh1outlook.onmicrosoft.com' THEN email ELSE '***@***.com' END;

### Task 3.2 — Apply the column mask to the email column

Apply the `mask_email` function as a column mask on the `email` column of the `customers` table.


# Apply the mask_email function as a column mask to the email column

spark.sql("""
    ALTER TABLE lab4.security_lab.customers
    ALTER COLUMN email SET MASK lab4.security_lab.mask_email
""")

Explanation:

- `ALTER TABLE` modifies an existing table.
- `lab4.security_lab.customers` is the table being updated.
- `ALTER COLUMN email` specifies that changes are being made to the `email` column.
- `SET MASK` applies a column masking function to that column.
- `lab4.security_lab.mask_email` is the masking function that will be executed whenever the email column is queried.

--------------------------------------------------

How it works:

When a user queries the `email` column:

- The `mask_email()` function is automatically called.
- Admin users see the actual email address.
- Non-admin users see the masked value `***@***.com`.

Example:

Original Value:
alice@northmart.com

Admin View:
alice@northmart.com

Regular User View:
***@***.com

--------------------------------------------------

Purpose:

Apply column-level security to the `email` column so sensitive customer email addresses are protected from unauthorized users.

--------------------------------------------------

Overall Purpose:

This command attaches the `mask_email` function to the `email` column, ensuring that email addresses are automatically masked for non-admin users while remaining visible to authorized administrators.

In [0]:
# Apply the mask_email function as a column mask to the email column
spark.sql("""
    ALTER TABLE dp750_lab_04.security_lab.customers
    ALTER COLUMN email SET MASK dp750_lab_04.security_lab.mask_email
""")

DataFrame[]

In [0]:
%sql
Select * from dp750_lab_04.security_lab.customers

customer_id,customer_name,email,region,total_spend,tier
1,Alice Johnson,alice@northmart.com,North,5200.00,Standard
2,Bob Martinez,bob@northmart.com,South,8100.00,Premium
3,Carol Lee,carol@northmart.com,North,3400.00,Standard
4,David Chen,david@northmart.com,East,7800.00,Premium
5,Eve Patel,eve@northmart.com,West,6200.00,Standard
6,Frank Kim,frank@northmart.com,South,4100.00,Standard


### Task 3.3 — Test the column mask

Query the `customers` table. Since you are the privileged user specified in the mask function, you should see the **real email addresses**.

Any other user querying this table would see `***@***.com` in the `email` column.

In [0]:
%sql
SELECT * FROM dp750_lab_04.security_lab.customers;

customer_id,customer_name,email,region,total_spend,tier
1,Alice Johnson,alice@northmart.com,North,5200.00,Standard
2,Bob Martinez,bob@northmart.com,South,8100.00,Premium
3,Carol Lee,carol@northmart.com,North,3400.00,Standard
4,David Chen,david@northmart.com,East,7800.00,Premium
5,Eve Patel,eve@northmart.com,West,6200.00,Standard
6,Frank Kim,frank@northmart.com,South,4100.00,Standard


### Task 3.4 — Remove the column mask

Remove the column mask from the `email` column, then query the table again to confirm emails are now fully visible to all users.



# Remove the column mask from the email column

spark.sql("""
    ALTER TABLE lab4.security_lab.customers
    ALTER COLUMN email DROP MASK
""")

Explanation:

- `ALTER TABLE` modifies an existing table.
- `lab4.security_lab.customers` is the table being updated.
- `ALTER COLUMN email` specifies that changes are being made to the `email` column.
- `DROP MASK` removes the masking function currently applied to the column.

--------------------------------------------------

What happens after this?

- The `mask_email()` function is no longer applied.
- Email addresses are returned in their original form.
- Users with `SELECT` permission can see the actual email values.

Example:

Before:
***@***.com

After:
alice@northmart.com

--------------------------------------------------

Purpose:

Remove the column-level security policy from the `email` column.

--------------------------------------------------

Overall Purpose:

This command disables email masking by removing the mask attached to the `email` column, allowing authorized users to view the original email addresses stored in the table.

In [0]:
# Remove the column mask from the email column
spark.sql("""
    ALTER TABLE dp750_lab_04.security_lab.customers
    ALTER COLUMN email DROP MASK
""")

DataFrame[]

In [0]:
%sql
ALTER TABLE dp750_lab_04.security_lab.customers
ALTER COLUMN email DROP MASK;

-- Query the table to confirm email addresses are now unmasked
SELECT * FROM dp750_lab_04.security_lab.customers;

customer_id,customer_name,email,region,total_spend,tier
1,Alice Johnson,alice@northmart.com,North,5200.00,Standard
2,Bob Martinez,bob@northmart.com,South,8100.00,Premium
3,Carol Lee,carol@northmart.com,North,3400.00,Standard
4,David Chen,david@northmart.com,East,7800.00,Premium
5,Eve Patel,eve@northmart.com,West,6200.00,Standard
6,Frank Kim,frank@northmart.com,South,4100.00,Standard


## Exercise 4: Azure Key Vault Secrets

NorthMart Retail's loyalty platform requires an API key to connect. Storing this key in plain text inside a notebook is a security risk. In this exercise you retrieve the API key securely from the Azure Key Vault-backed secret scope you created during lab setup.

> ⚠️ Before running the cells in this exercise, ensure you have completed the Key Vault setup steps in the lab instructions (create Key Vault, add secret `loyalty-api-key`, create secret scope `retail-kv-scope`).

### Task 4.1 — List available secrets in the scope

List all secrets available in the `retail-kv-scope` secret scope to confirm that `loyalty-api-key` is accessible.

> 🤖 **Genie Code tip:** Ask:
> *"How do I list the secrets in a Databricks secret scope using dbutils?"*

**Hint:** Use `dbutils.secrets.list("scope-name")`. This returns key names only — never secret values.

# Make sure you have completed the AZure key vault development

In [0]:
# List all secrets in the retail-kv-scope and display the output
display(dbutils.secrets.list("dp_750keyscope"))

key
loyalty-api-key


### Task 4.2 — Retrieve the API key securely in Python

Retrieve the `loyalty-api-key` secret from the `retail-kv-scope` scope and assign it to a variable. Then print it.

Observe how Azure Databricks **automatically redacts** the value in the notebook output — you will see `[REDACTED]` instead of the actual key.



In [0]:
# Retrieve the loyalty-api-key from the secret scope
api_key = dbutils.secrets.get(scope="dp_750keyscope", key="loyalty-api-key")

# Print the api_key variable — output will show [REDACTED]
print(api_key)

[REDACTED]


### Task 4.3 — Use the secret to simulate a connection string

Construct a connection string that embeds the API key, simulating how it would be used to connect to NorthMart's loyalty platform. Print the connection string and observe that the secret value is still redacted.



In [0]:
# Build a connection string using the api_key variable
connection_string = f"https://loyalty.northmart.com/api?key={api_key}"

# Print the connection string — secret value will be redacted in output
print(connection_string)

https://loyalty.northmart.com/api?key=[REDACTED]


### Task 4.4 — Retrieve the secret using SQL

Retrieve the same secret using the `secret()` SQL function. This is useful when constructing SQL-based connections.

Note that the output is also redacted in SQL results.



In [0]:
# Use Databricks SQL to retrieve the loyalty-api-key secret from the dp_750keyscope
query = """
SELECT secret('dp_750keyscope', 'loyalty-api-key') AS api_key
"""
df = spark.sql(query)
display(df)

api_key
[REDACTED]


## Cleanup Optional

Run the cell below to remove all objects created during this lab.

In [0]:
%sql
-- Remove all lab objects
ALTER TABLE dp750_lab_04.security_lab.customers DROP ROW FILTER;
ALTER TABLE dp750_lab_04.security_lab.customers ALTER COLUMN email DROP MASK;
DROP TABLE IF EXISTS dp750_lab_04.security_lab.customers;
DROP FUNCTION IF EXISTS dp750_lab_04.security_lab.region_filter;
DROP FUNCTION IF EXISTS dp750_lab_04.security_lab.mask_email;
DROP SCHEMA IF EXISTS dp750_lab_04.security_lab;